# ProtRL Quick Start: Reinforcement Learning for Protein Language Models

This notebook shows how to use **ProtRL** to steer a protein language model with Reinforcement Learning — in just a few lines of code.

**Model:** `AI4PD/ProtGPT3-112M` (112M parameters, fits on a free Colab T4 GPU)

**Algorithm:** GRPO (offline)

**LoRA:** only ~0.5% of parameters are trained

**Objective:** steer the model to generate sequences of exactly **100 amino acids**
```
reward = -|len(sequence) - 100|
```
No external tools — just the model and a reward function.

---
> **Before running:** set the runtime to GPU → *Runtime → Change runtime type → T4 GPU*

## 1. Setup

In [ ]:
# Install dependencies
!pip install transformers datasets peft accelerate trl --quiet

# Clone ProtRL if running in Colab
import sys, os
if not os.path.exists('src/pLM_GRPO.py'):
    !git clone https://github.com/AI4PDLab/ProtRL.git
    sys.path.insert(0, 'ProtRL')

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model, TaskType
from datasets import Dataset
import matplotlib.pyplot as plt

from src.ProtRL_Trainer import ProtRLTrainingArgument
from src.pLM_GRPO import ProtRL_GRPOTrainer

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

## 2. Load Model and Add LoRA

We load ProtGPT3-112M and wrap it with a LoRA adapter. This means only ~0.5% of weights are trained — ideal for a single GPU.

In [ ]:
MODEL_NAME = "AI4PD/ProtGPT3-112M"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# ProtRL requires distinct pad, bos, and eos tokens.
# GPT2-style protein models often lack a pad token — we add one.
if tokenizer.pad_token is None:
    tokenizer.add_special_tokens({'pad_token': '<pad>'})
if tokenizer.bos_token is None:
    tokenizer.bos_token = tokenizer.eos_token
    tokenizer.bos_token_id = tokenizer.eos_token_id

tokenizer.padding_side = "left"  # needed for batch generation

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16,
)
model.resize_token_embeddings(len(tokenizer))  # register new pad token
model = model.to(device)

print(f"Total parameters: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules="all-linear",
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## 3. (Optional) Supervised Fine-Tuning

Before RL, you can warm-start the LoRA adapter with a short **supervised fine-tuning (SFT)** pass on known sequences. SFT teaches the model the target distribution with standard cross-entropy; RL then steers it toward the reward objective.

This step is **optional** — RL alone works directly from the pretrained model.

> Replace `SFT_SEQUENCES` with your own curated dataset (e.g., sequences from a specific protein family, thermostable variants, etc.).

In [ ]:
from trl import SFTConfig, SFTTrainer

# ── Example dataset ────────────────────────────────────────────────────────────
# Replace with your own sequences (FASTA file, HuggingFace dataset, CSV, etc.)
SFT_SEQUENCES = [
    "MKVLSPADKTNVKAAWGKVGAHAGEYGAEALERMFLSFPTTKTYFPHFDLSHGSAQVKGHGKKVADALTNAVAHVDDMPNALSALSDLHAHKLRVDPVNFKLLSHCLLVTLAAHLPAEFTPAVHASLDKFLASVSTVLTSKYR",
    "MGSSHHHHHHSSGLVPRGSHMASMTGGQQMGRDLYDDDDKDRWGSHMSSSVPSQKTYPGDLNYLDAGKSGLFKDLQKKLKGGLKQRVDWKTLRDLNQGFQPNLNQTRETVKYFLKEKNAEVLAYGDFQVHSGDFIAGLDNYTC",
    "MTEYKLVVVGAGGVGKSALTIQLIQNHFVDEYDPTIEDSY YRDQILRVKDSEDVPMVLVGNKCDLPSRTVESRQAQDLARSYGIPYIETSAKTRQHVREVDRE",
    "MASQSLTKEQIEKLKQMTQEEYQMLQQQLEQLQGQDQPTLRPPATITPLSSSPEEDTPKKKRKVHHHHHH",
    "MHHHHHHSSGVDLGTENLYFQSNAHSISVKVSDDGKFSVLKGDKVNLHFSMPEGKFKIAHDFDSIETGKL",
]

sft_dataset = Dataset.from_dict({"text": SFT_SEQUENCES})

# ── SFT training ──────────────────────────────────────────────────────────────
sft_config = SFTConfig(
    output_dir="./sft_warmup",
    num_train_epochs=5,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=2,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    logging_steps=5,
    save_strategy="no",
    report_to="none",
    bf16=True,
    max_seq_length=256,
)

sft_trainer = SFTTrainer(
    model=model,          # LoRA adapter already applied
    processing_class=tokenizer,
    args=sft_config,
    train_dataset=sft_dataset,
)

print("Starting SFT warm-up...")
sft_trainer.train()
print("SFT complete — model is ready for RL fine-tuning.")

## 4. Define the RL Loop

ProtRL uses an **offline** RL loop:

```
for each iteration:
    1. generate sequences with the current model
    2. score each sequence with the reward function
    3. train the model on the (sequence, reward) pairs
```

This decouples scoring from training — scoring can be done on CPUs or external servers.

In [ ]:
VALID_AA = set("ACDEFGHIKLMNPQRSTVWY")

def generate_sequences(model, tokenizer, n_sequences=32, max_new_tokens=200):
    """Generate protein sequences unconditionally from the current model."""
    model.eval()

    # Empty prompt for unconditional generation
    inputs = tokenizer("", return_tensors="pt", add_special_tokens=True).to(device)

    with torch.no_grad():
        output_ids = model.generate(
            input_ids=inputs["input_ids"].repeat(n_sequences, 1),
            attention_mask=inputs["attention_mask"].repeat(n_sequences, 1),
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=1.0,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    prompt_len = inputs["input_ids"].shape[1]
    sequences = []
    for ids in output_ids:
        text = tokenizer.decode(ids[prompt_len:], skip_special_tokens=True)
        # Strip non-amino-acid characters (handles space-separated tokenizers)
        seq = "".join(c for c in text if c in VALID_AA)
        sequences.append(seq)

    return sequences

In [ ]:
TARGET_LENGTH = 100  # amino acids

def reward_fn(sequences):
    """Reward sequences that are close to TARGET_LENGTH amino acids."""
    # Negative distance to the target length: 0 is the best possible score
    # (exactly TARGET_LENGTH AA) and it gets more negative the further off you are.
    # GRPO maximizes the reward, so this still pushes the model toward TARGET_LENGTH
    # even though every value is <= 0 — see the "higher is better" note in the README.
    return [-abs(len(seq) - TARGET_LENGTH) for seq in sequences]

# Quick sanity check
test_seqs = ["A" * 100, "A" * 50, "A" * 200]
print("Rewards:", reward_fn(test_seqs))  # [0, -50, -100]

In [ ]:
def run_rl_iteration(model, tokenizer, iteration, n_sequences=32):
    """One RL iteration: generate → score → train."""

    # 1. Generate sequences
    completions = generate_sequences(model, tokenizer, n_sequences=n_sequences)

    # 2. Score
    rewards = reward_fn(completions)

    # 3. Build dataset
    # All sequences share the same prompt → they form one preference set
    dataset = Dataset.from_dict({
        "prompt":     [""] * len(completions),
        "completion": completions,
        "reward":     [float(r) for r in rewards],
    })

    # 4. Train
    args = ProtRLTrainingArgument(
        output_dir=f"./rl_run/iter_{iteration}",
        num_train_epochs=1,
        per_device_train_batch_size=8,
        beta=0.05,          # KL penalty weight
        logging_steps=5,
        save_strategy="no",
        report_to="none",
        bf16=True,
    )

    trainer = ProtRL_GRPOTrainer(
        model=model,                # already has LoRA; no peft_config needed
        processing_class=tokenizer,
        args=args,
        train_dataset=dataset,
    )
    trainer.train()

    mean_reward = sum(rewards) / len(rewards)
    mean_len    = sum(len(s)   for s in completions) / len(completions)
    print(f"[iter {iteration:2d}]  mean reward = {mean_reward:6.1f}  |  mean length = {mean_len:.0f} AA")

    return mean_reward, mean_len

## 5. Run the RL Training Loop

10 iterations, ~32 sequences each. The model should progressively shift towards sequences closer to 100 amino acids.

In [ ]:
N_ITERATIONS = 10
N_SEQUENCES  = 32

rewards_history = []
lengths_history = []

# Baseline: measure before any RL training
print("Measuring baseline...")
baseline_seqs = generate_sequences(model, tokenizer, n_sequences=N_SEQUENCES)
baseline_len  = sum(len(s) for s in baseline_seqs) / len(baseline_seqs)
print(f"Baseline mean length: {baseline_len:.0f} AA")

# RL loop
for i in range(N_ITERATIONS):
    mean_r, mean_l = run_rl_iteration(model, tokenizer, i, n_sequences=N_SEQUENCES)
    rewards_history.append(mean_r)
    lengths_history.append(mean_l)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(rewards_history, marker='o', color='steelblue', linewidth=2)
axes[0].axhline(0, color='crimson', linestyle='--', label='Max reward (perfect)')
axes[0].set_xlabel("RL iteration")
axes[0].set_ylabel("Mean reward")
axes[0].set_title("Reward over RL iterations")
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].axhline(baseline_len, color='gray', linestyle=':', label=f'Baseline ({baseline_len:.0f} AA)')
axes[1].plot(lengths_history, marker='o', color='darkorange', linewidth=2)
axes[1].axhline(TARGET_LENGTH, color='crimson', linestyle='--', label=f'Target ({TARGET_LENGTH} AA)')
axes[1].set_xlabel("RL iteration")
axes[1].set_ylabel("Mean sequence length (AA)")
axes[1].set_title("Sequence length over RL iterations")
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.suptitle(f"ProtRL — {MODEL_NAME}", fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig("rl_training_curve.png", dpi=150, bbox_inches='tight')
plt.show()
print("Saved to rl_training_curve.png")

In [ ]:
# Save the trained LoRA adapter
model.save_pretrained("ProtGPT3_length100_lora")
tokenizer.save_pretrained("ProtGPT3_length100_lora")
print("Adapter saved to ProtGPT3_length100_lora/")

---
## Summary

| Step | What happens |
|---|---|
| Load model | ProtGPT3-112M loaded in bfloat16 |
| Add LoRA | Only ~0.5% of parameters are trained |
| Generate | Model samples 32 sequences per iteration |
| Score | Reward = how close to 100 AA |
| Train | GRPO update: advantage-weighted log-prob + KL penalty |
| Repeat | Model improves over 10 iterations |

To use a different reward — hydrophobicity, secondary structure content, predicted stability — just replace `reward_fn`. Everything else stays the same.

## References

- Stocco et al. *Guiding Generative Protein Language Models with Reinforcement Learning*, arXiv 2412.12979 (2024)
- Garibbo et al. *ProtGPT3: an Open-source family of Promptable and Aligned Protein Language Models*, bioRxiv (2026)
- Ferruz et al. *ProtGPT2 is a deep unsupervised language model for protein design*, Nat Commun (2022)
- Hu et al. *LoRA: Low-Rank Adaptation of Large Language Models*, ICLR (2022)
- Shao et al. *DeepSeekMath: Pushing the Limits of Mathematical Reasoning in Open Language Models*, arXiv (2024) — GRPO